# Machine Learning: Supervised Learning after Anomaly Removal
**Copyright © 2026 Sujeet Banerjee**  
**Email:** sujee.banerjee@gmail.com

---

Welcome to this advanced beginner workbook! In this session, we will combine Supervised and Unsupervised learning:
1. We will train a **Baseline Linear Regression** model on the raw California housing data.
2. We will use an **Unsupervised Model (One-Class SVM)** to find and remove anomalies (outliers) from our training data.
3. We will train a **New Linear Regression** model on this newly cleaned dataset.

**The Goal:** To see if removing the weirdest 5% of our training data helps the Linear Regression model make better, more accurate predictions on the test set (measured by a lower RMSE).

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from sklearn.svm import OneClassSVM
from sklearn.preprocessing import StandardScaler

# 1. Load the data
print("1. Loading California Housing Data...")
housing = fetch_california_housing(as_frame=True)
X = housing.data
y = housing.target

# 2. Split into Training and Testing sets FIRST
# RULE: We only remove outliers from the TRAINING set. 
# The test set must represent the real world, which includes weird data!
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Original Training Set Size: {X_train.shape[0]} districts")

1. Loading California Housing Data...
Original Training Set Size: 16512 districts


### Step 3: Train the Baseline Model (With Outliers)
First, let's train a model on the raw, unfiltered data so we have a score to beat.

In [2]:
# Train baseline model
baseline_model = LinearRegression()
baseline_model.fit(X_train, y_train)

# Predict and evaluate on the test set
baseline_predictions = baseline_model.predict(X_test)
baseline_rmse = np.sqrt(mean_squared_error(y_test, baseline_predictions))

print(f"BASELINE RMSE (Trained on ALL data): {baseline_rmse:.4f}")

BASELINE RMSE (Trained on ALL data): 0.7456


### Step 4: Remove Anomalies using One-Class SVM
Now we use our unsupervised learning technique to find the 5% of training data that looks the most unusual.

In [3]:
print("Scaling data and finding anomalies in the training set...\n")

# Scale the features (Required for SVM)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)

# Initialize and fit the One-Class SVM (nu=0.05 asks for ~5% outliers)
svm = OneClassSVM(kernel='rbf', gamma='scale', nu=0.05)
outlier_labels = svm.fit_predict(X_train_scaled)

# Filter the training data
# Keep only the rows where the SVM predicted '1' (Normal)
mask = (outlier_labels == 1)
X_train_clean = X_train[mask]
y_train_clean = y_train[mask]

print(f"Removed {(~mask).sum()} anomalous districts.")
print(f"Cleaned Training Set Size: {X_train_clean.shape[0]} districts")

Scaling data and finding anomalies in the training set...

Removed 825 anomalous districts.
Cleaned Training Set Size: 15687 districts


### Step 5: Train the New Model on Clean Data
Let's see if training on this smaller, but "cleaner" dataset improves our predictions on the test set.

In [4]:
# Train new model on CLEAN data
clean_model = LinearRegression()
clean_model.fit(X_train_clean, y_train_clean)

# Predict and evaluate on the SAME test set as before
clean_predictions = clean_model.predict(X_test)
clean_rmse = np.sqrt(mean_squared_error(y_test, clean_predictions))

print("--- RESULTS ---")
print(f"BASELINE RMSE: {baseline_rmse:.4f}")
print(f"CLEANED RMSE:  {clean_rmse:.4f}")

if clean_rmse < baseline_rmse:
    print(f"\nSuccess! Removing anomalies reduced our error by {baseline_rmse - clean_rmse:.4f}")
else:
    print(f"\nInteresting! Removing anomalies actually increased our error by {clean_rmse - baseline_rmse:.4f}")

--- RESULTS ---
BASELINE RMSE: 0.7456
CLEANED RMSE:  0.9472

Interesting! Removing anomalies actually increased our error by 0.2016
